# MAE (Masked Autoencoder) Analysis

Comprehensive analysis of a Masked Autoencoder pretrained on STL-10 (96x96 images).
MAE is ViT-based (not CNN), so visualization differs from the contrastive/reconstruction methods.

**Sections:**
- **A. Training Trajectories** -- loss, k-NN accuracy, and learning rate over epochs
- **B. Evaluation Results** -- k-NN sweep and linear probe (full + low-data)
- **C. Attention Map Visualization** -- per-head attention from the ViT encoder
- **D. Masked Reconstruction Visualization** -- original, masked, and reconstructed images

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from models.mae import MAE
from evaluation import extract_features
from evaluation.knn import knn_classify
from evaluation.linear_probe import LinearProbe
from utils.data import get_eval_loaders, STL10_MEAN, STL10_STD

sns.set_theme(style="whitegrid", font_scale=1.1)
device = "cuda" if torch.cuda.is_available() else "cpu"
CLASS_NAMES = ["airplane", "bird", "car", "cat", "deer", "dog", "horse", "monkey", "ship", "truck"]
RESULTS_DIR = PROJECT_ROOT / "results" / "mae"

print(f"Device: {device}")
print(f"Results dir: {RESULTS_DIR}")

## A. Training Trajectories

Load the training history and plot loss, k-NN accuracy, and learning rate schedule over epochs.

In [ ]:
history_path = RESULTS_DIR / "history.json"
if history_path.exists():
    with open(history_path) as f:
        history = json.load(f)

    epochs = [h["epoch"] for h in history]
    losses = [h["loss"] for h in history]
    knn_accs = [h["knn_top1"] for h in history]
    lrs = [h["lr"] for h in history]

    print(f"Training epochs: {len(history)}")
    print(f"Final loss: {losses[-1]:.4f}")
    print(f"Best k-NN accuracy: {max(knn_accs):.4f} (epoch {epochs[np.argmax(knn_accs)]})")
else:
    print("history.json not found \u2014 training may still be in progress.")

In [ ]:
if history_path.exists():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Loss curve
    axes[0].plot(epochs, losses, color="steelblue")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("MAE Training Loss (MSE Reconstruction)")

    # k-NN accuracy curve
    axes[1].plot(epochs, [a * 100 for a in knn_accs], color="darkorange")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("k-NN Accuracy (%)")
    axes[1].set_title("k-NN Top-1 Accuracy During Training")

    # Learning rate schedule
    axes[2].plot(epochs, lrs, color="seagreen")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Learning Rate")
    axes[2].set_title("Learning Rate Schedule")

    plt.tight_layout()
    plt.show()

## B. Evaluation Results

Load the trained MAE model, extract features using the ViT encoder (average-pooled patch embeddings),
and evaluate with k-NN and linear probe.

In [ ]:
# Load pre-computed evaluation results across all checkpoints
all_eval_path = PROJECT_ROOT / "results" / "all_eval_results.json"
if all_eval_path.exists():
    with open(all_eval_path) as f:
        all_results = json.load(f)

    mae_results = sorted(
        [r for r in all_results if r["method"] == "mae" and r["checkpoint_name"].startswith("checkpoint_")],
        key=lambda r: r["epoch"],
    )
    # Include best/last if no periodic checkpoints
    if not mae_results:
        mae_results = sorted(
            [r for r in all_results if r["method"] == "mae"],
            key=lambda r: r["epoch"],
        )
    print(f"Loaded {len(mae_results)} checkpoint evaluations")
else:
    mae_results = []
    print("all_eval_results.json not found — run scripts/evaluate_all.py first")

In [ ]:
if mae_results:
    eval_epochs = [r["epoch"] for r in mae_results]
    eval_knn20 = [r["knn"]["20"] for r in mae_results]
    eval_probe = [r["linear_probe"] for r in mae_results]
    eval_probe_low = [r["linear_probe_lowdata"] for r in mae_results]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(eval_epochs, [a * 100 for a in eval_knn20], "o-", color="steelblue")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy (%)")
    axes[0].set_title("k-NN Accuracy (k=20)")

    axes[1].plot(eval_epochs, [a * 100 for a in eval_probe], "o-", color="darkorange")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
    axes[1].set_title("Linear Probe (full data)")

    axes[2].plot(eval_epochs, [a * 100 for a in eval_probe_low], "o-", color="seagreen")
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Accuracy (%)")
    axes[2].set_title("Linear Probe (1% data)")

    fig.suptitle("MAE — Evaluation Across Training", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
if mae_results:
    best_idx = int(np.argmax(eval_probe))
    best_result = mae_results[best_idx]
    best_knn = best_result["knn"]

    fig, ax = plt.subplots(figsize=(8, 5))
    k_vals = [int(k) for k in best_knn.keys()]
    accs = [best_knn[str(k)] * 100 for k in k_vals]
    ax.plot(k_vals, accs, "o-", color="steelblue")
    ax.set_xscale("log"); ax.set_xlabel("k"); ax.set_ylabel("Accuracy (%)")
    ax.set_title(f"k-NN Accuracy vs k (epoch {best_result['epoch']})")
    plt.tight_layout()
    plt.show()

In [ ]:
if mae_results:
    # Summary table
    print(f"{'Checkpoint':<25s} {'Epoch':>6s} {'k-NN(20)':>10s} {'Probe':>10s} {'Probe(1%)':>10s}")
    print("-" * 63)
    for r in mae_results:
        marker = " <-- best" if r is best_result else ""
        print(f"{r['checkpoint_name']:<25s} {r['epoch']:>6d} "
              f"{r['knn']['20']*100:>9.2f}% {r['linear_probe']*100:>9.2f}% "
              f"{r['linear_probe_lowdata']*100:>9.2f}%{marker}")

## C. Attention Map Visualization

The MAE encoder is a custom ViT with 6 TransformerBlocks. Each block uses `nn.MultiheadAttention`.
We monkey-patch selected blocks to capture per-head attention weights, then visualize
how different heads attend from a center query patch across the 24x24 patch grid.

In [ ]:
# Load model (best by linear probe if available, else best.pt) for visualization
if mae_results:
    best_ckpt_path = RESULTS_DIR / best_result["checkpoint_name"]
else:
    best_ckpt_path = RESULTS_DIR / "best.pt"

ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=True)
ckpt_args = ckpt["args"]
model = MAE(
    patch_size=ckpt_args.get("patch_size", 4),
    encoder_dim=ckpt_args.get("encoder_dim", 384),
    encoder_depth=ckpt_args.get("encoder_depth", 6),
    encoder_heads=ckpt_args.get("encoder_heads", 6),
    decoder_dim=ckpt_args.get("decoder_dim", 192),
    decoder_depth=ckpt_args.get("decoder_depth", 4),
    decoder_heads=ckpt_args.get("decoder_heads", 3),
    mask_ratio=ckpt_args.get("mask_ratio", 0.75),
)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device).eval()

loaders = get_eval_loaders(batch_size=256)
print(f"Loaded MAE from {best_ckpt_path.name}")

In [ ]:
attn_maps = {}


def patch_block(block, layer_idx):
    """Monkey-patch a TransformerBlock to capture attention weights."""
    original_forward = block.forward

    def new_forward(x):
        normed = block.norm1(x)
        attn_out, attn_weights = block.attn(
            normed, normed, normed,
            need_weights=True, average_attn_weights=False,
        )
        attn_maps[layer_idx] = attn_weights.detach().cpu()  # (B, num_heads, N, N)
        x = x + attn_out
        x = x + block.mlp(block.norm2(x))
        return x

    block.forward = new_forward
    return original_forward


# Patch blocks 0, 2, 5 (first, middle, last)
originals = {}
for i in [0, 2, 5]:
    originals[i] = patch_block(model.encoder.blocks[i], i)

# Run forward on a single image
sample_img = next(iter(loaders["test"]))[0][:1].to(device)
with torch.no_grad():
    model.encoder(sample_img)

# Restore originals
for i, orig in originals.items():
    model.encoder.blocks[i].forward = orig

print(f"Captured attention maps for blocks: {sorted(attn_maps.keys())}")
for idx, attn in attn_maps.items():
    print(f"  Block {idx}: {tuple(attn.shape)}")

In [ ]:
# Visualize attention from the center query patch
# 576 = 24x24 patches (patch_size=4, image=96)
center = 24 * 12 + 12  # = 300

for layer_idx in [0, 2, 5]:
    attn = attn_maps[layer_idx][0]  # (6, 576, 576)
    fig, axes = plt.subplots(1, 6, figsize=(18, 3))
    fig.suptitle(f"Block {layer_idx} \u2014 Attention from center patch", fontsize=14)
    for head in range(6):
        head_attn = attn[head, center].reshape(24, 24)
        axes[head].imshow(head_attn.numpy(), cmap="viridis")
        axes[head].set_title(f"Head {head}")
        axes[head].axis("off")
    plt.tight_layout()
    plt.show()

## D. Masked Reconstruction Visualization

Run the full MAE forward pass (mask, encode visible patches, decode) and visualize
the original images, what the model sees (masked input), and the reconstructions.

In [ ]:
# Run the full MAE forward (masks, encodes visible, decodes)
batch = next(iter(loaders["test"]))[0][:4].to(device)
with torch.no_grad():
    loss, pred, mask = model(batch)

print(f"Reconstruction loss: {loss.item():.4f}")
print(f"pred shape: {tuple(pred.shape)}")   # (4, 576, 48)
print(f"mask shape: {tuple(mask.shape)}")    # (4, 576)

In [ ]:
def unpatchify(patches, patch_size=4, img_size=96):
    """Convert patches back to images: (B, N, patch_size^2*3) -> (B, 3, H, W)."""
    p = patch_size
    h = w = img_size // p
    B = patches.shape[0]
    patches = patches.reshape(B, h, w, p, p, 3)
    return patches.permute(0, 5, 1, 3, 2, 4).reshape(B, 3, img_size, img_size)


def denormalize(tensor, mean=STL10_MEAN, std=STL10_STD):
    """Undo ImageNet-style normalization for display."""
    t = tensor.clone()
    for i, (m, s) in enumerate(zip(mean, std)):
        t[:, i].mul_(s).add_(m)
    return t.clamp(0, 1)


# Build the three views: original, masked, reconstructed
target_patches = model.patchify(batch).cpu()
masked_patches = target_patches.clone()
masked_patches[mask.cpu()] = 0  # zero out masked patches

orig_img = denormalize(batch.cpu())
masked_img = denormalize(unpatchify(masked_patches))
recon_img = denormalize(unpatchify(pred.cpu()))

print(f"Original image range:  [{orig_img.min():.3f}, {orig_img.max():.3f}]")
print(f"Masked image range:    [{masked_img.min():.3f}, {masked_img.max():.3f}]")
print(f"Recon image range:     [{recon_img.min():.3f}, {recon_img.max():.3f}]")

In [ ]:
# Show 3 rows (original, masked, reconstructed) x 4 samples
n_samples = 4
fig, axes = plt.subplots(3, n_samples, figsize=(4 * n_samples, 12))
row_labels = ["Original", "Masked (visible patches)", "Reconstruction"]

for col in range(n_samples):
    axes[0, col].imshow(orig_img[col].permute(1, 2, 0).numpy())
    axes[1, col].imshow(masked_img[col].permute(1, 2, 0).numpy())
    axes[2, col].imshow(recon_img[col].permute(1, 2, 0).numpy())

    for row in range(3):
        axes[row, col].axis("off")

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=13, fontweight="bold", rotation=90, labelpad=10)
    axes[row, 0].yaxis.set_visible(True)
    axes[row, 0].set_yticks([])

fig.suptitle(f"MAE Masked Reconstruction (mask ratio: {ckpt_args.get('mask_ratio', 0.75):.0%})",
             fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()